<a id="inicio"></a>
<br><br><br>
<div align="left">
<h1><font size=6>Pandas</font></h1>
    <h2><font color="#5E5E5E" size=5>Ejercicios II</h2>
<br><br><br><br>
</div>


<div align="right">
<font color="#B51700" size=4><b>Análisis y visualización de datos</b></font><br><br>
<font  size=3>Máster Universitario en Ingeniería Informática</font><br>
<font  size=3>Máster Universitario en Big Data y Computación en la Nube</font><br>
<font size=3>Máster Universitario en Inteligencia Artificial</font><br><br>
<font color="#5E5E5E" size=3><b>Luis de la Ossa  &nbsp;&nbsp;</b></font><span style="background-color:#B51700; color:white; padding:2px 6px; font-family:Arial, sans-serif;font-weight:bold;font-size:18px;">UCLM</span>
</div>

---

<a id="indice"></a>
<h2><font color="#5E5E5E" size=5>Índice</font></h2>


* [9. Acceso condicional a datos](#section9)
    * [9.1 Selección condicional](#section91)          
    * [9.2 Filtrado por valor del índice](#section92) 
    * [9.3 Reemplazo condicional de valores](#section93)
* [10. Agrupaciones](#section10)
    * [10.1 Agrupación por categorías mediante `DataFrame.groupby()`](#section101)
    * [10.7 Agrupación por categorías externas. Grupos a partir de datos numéricos](#section103)        
* [11. Operaciones sobre grupos](#section11)        
    * [11.1 Agregación en grupos](#section111)     
    * [11.2 Filtrado por grupos](#section112)     
    * [11.3 Transformación por grupo](#section113)    
    * [11.4 `DataFrameGroupBy.apply()`](#section114)  
* [13. Combinación de DataFrames](#section13)
    * [13.2 Concatenación de *Series* y *DataFrames*](#section131)
    * [13.3 Fusión de *DataFrames*](#section132)
* [14. Índices de varios niveles](#section14)   
* [15. Tablas dinámicas y reestructuración](#section150)
    * [15.1 Creación de tablas dinámicas](#section151)
    * [15.2 Reestructuración de tablas](#section152)

---

Para esta serie de ejercicios se utilizará el conjunto de datos `fifa.csv`. 

In [164]:
import pandas as pd
import numpy as np

# Lee un archivo .csv y utiliza el campo ID como índice.
df_fifa = pd.read_csv('./data/fifa19.csv', index_col=0).set_index('ID')

# Muestra la cabecera (5 primeras filas)
df_fifa.head()

,Name,Age,Photo,Nationality,Flag,Overall,Potential,Club,Club Logo,Value,...,Composure,Marking,StandingTackle,SlidingTackle,GKDiving,GKHandling,GKKicking,GKPositioning,GKReflexes,Release Clause
ID,,,,,,,,,,,,,,,,,,,,,
158023,L. Messi,31,https://cdn.sofifa.org/players/4/19/158023.png,Argentina,https://cdn.sofifa.org/flags/52.png,94,94,FC Barcelona,https://cdn.sofifa.org/teams/2/light/241.png,€110.5M,...,96.0,33.0,28.0,26.0,6.0,11.0,15.0,14.0,8.0,€226.5M
20801,Cristiano Ronaldo,33,https://cdn.sofifa.org/players/4/19/20801.png,Portugal,https://cdn.sofifa.org/flags/38.png,94,94,Juventus,https://cdn.sofifa.org/teams/2/light/45.png,€77M,...,95.0,28.0,31.0,23.0,7.0,11.0,15.0,14.0,11.0,€127.1M
190871,Neymar Jr,26,https://cdn.sofifa.org/players/4/19/190871.png,Brazil,https://cdn.sofifa.org/flags/54.png,92,93,Paris Saint-Germain,https://cdn.sofifa.org/teams/2/light/73.png,€118.5M,...,94.0,27.0,24.0,33.0,9.0,9.0,15.0,15.0,11.0,€228.1M
193080,De Gea,27,https://cdn.sofifa.org/players/4/19/193080.png,Spain,https://cdn.sofifa.org/flags/45.png,91,93,Manchester United,https://cdn.sofifa.org/teams/2/light/11.png,€72M,...,68.0,15.0,21.0,13.0,90.0,85.0,87.0,88.0,94.0,€138.6M
192985,K. De Bruyne,27,https://cdn.sofifa.org/players/4/19/192985.png,Belgium,https://cdn.sofifa.org/flags/7.png,91,92,Manchester City,https://cdn.sofifa.org/teams/2/light/10.png,€102M,...,88.0,68.0,58.0,51.0,15.0,13.0,5.0,10.0,13.0,€196.4M


A continuación, se almacenan en la lista `sel_columns` los nombres de las columnas de interés, y se eliminan las filas que tienen un `NaN` en alguna de ellas.

In [3]:
# Lista con los nombres de las columnas de interés
sel_columns = ['Name', 'Age', 'Nationality', 'Overall', 
               'Potential', 'Club', 'Value', 'Wage', 'Position', 
               'Joined', 'Height', 'Weight', 'Release Clause']

df_fifa.dropna(subset=sel_columns, how='any', axis=0, inplace=True)

Se preprocesan las columnas que representan valores numéricos.

In [4]:
feet_to_cm = 30.48 # Pies a centímetros
inch_to_cm = 2.54  # Pulgadas a centrímetros

# Toma un String con formato "pies'pulgadas" y devuelve un número con los metros
def imp_to_metric(imp_height):
    feet, inch = imp_height.split("'")                             # Separa pies y pulgadas 
    return (float(feet)*feet_to_cm + float(inch)*inch_to_cm)/100   # Convierte a float y calcula los metros

# Convierte toda la columna utilizando la función `imp_to_metric`.
df_fifa['Height'] = df_fifa['Height'].map(imp_to_metric)    
df_fifa['Weight'] = df_fifa['Weight'].map(lambda w: float(w[:-3])*0.453592) # Traduce a kilos la columna.
df_fifa['Value'] = df_fifa['Value'].map(lambda v: float(v[1:-1]))           # Convierte en numérico.
df_fifa['Wage'] = df_fifa['Wage'].map(lambda w: int(w[1:-1])*1000)          # Convierte a numérica la ganancia semanal.

Corrige el nombre del `Atlético de Madrid`.

In [7]:
df_fifa['Club'] = df_fifa['Club'].replace('Atlético Madrid','Atlético de Madrid')

Crea el *DataFrame* `df_fifa_att` con las columnas que no están recogidas en `sel_columns`.

In [120]:
df_fifa_att = df_fifa.drop(columns=sel_columns)
df_fifa_att.head()

""
ID
158023
20801
190871
193080
192985


Deja el el *DataFrame* `df_fifa` las columnas correspondientes a `sel_columns`.

In [8]:
df_fifa = df_fifa[sel_columns].copy()
df_fifa.head()

,Name,Age,Nationality,Overall,Potential,Club,Value,Wage,Position,Joined,Height,Weight,Release Clause
ID,,,,,,,,,,,,,
158023,L. Messi,31,Argentina,94,94,FC Barcelona,110.5,565000,RF,"Jul 1, 2004",1.7018,72.121128,€226.5M
20801,Cristiano Ronaldo,33,Portugal,94,94,Juventus,77.0,405000,ST,"Jul 10, 2018",1.8796,83.007336,€127.1M
190871,Neymar Jr,26,Brazil,92,93,Paris Saint-Germain,118.5,290000,LW,"Aug 3, 2017",1.7526,68.038800,€228.1M
193080,De Gea,27,Spain,91,93,Manchester United,72.0,260000,GK,"Jul 1, 2011",1.9304,76.203456,€138.6M
192985,K. De Bruyne,27,Belgium,91,92,Manchester City,102.0,355000,RCM,"Aug 30, 2015",1.8034,69.853168,€196.4M


<div align="right">
<a href="#indice"><font size=5 color=#B51700><i class="fa fa-arrow-circle-up" aria-hidden="true" ></i></font></a>
</div>

---

<a id="section9"></a>
## <font color="#5E5E5E"> 9. Acceso condicional a datos </font>



<a id="section91"></a> 
### <font color="#5E5E5E">9.1 Selección condicional</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 1</font> 

Mostrar las filas correspondientes a los jugadores cuyo club es `Atlético de Madrid`. Utilizar selección condicional.

In [11]:
df_fifa[df_fifa['Club'] == 'Atlético de Madrid']

,Name,Age,Nationality,Overall,Potential,Club,Value,Wage,Position,Joined,Height,Weight,Release Clause
ID,,,,,,,,,,,,,
200389,J. Oblak,25,Slovenia,90,93,Atlético de Madrid,68.0,94000,GK,"Jul 16, 2014",1.8796,87.089664,€144.5M
182493,D. Godín,32,Uruguay,90,90,Atlético de Madrid,44.0,125000,CB,"Aug 4, 2010",1.8796,78.017824,€90.2M
194765,A. Griezmann,27,France,89,90,Atlético de Madrid,78.0,145000,CAM,"Jul 28, 2014",1.7526,73.028312,€165.8M
208421,Saúl,23,Spain,85,90,Atlético de Madrid,50.5,77000,RCM,"Jul 1, 2013",1.8288,77.110640,€107.3M
193747,Koke,26,Spain,85,86,Atlético de Madrid,45.0,88000,LM,"Jan 1, 2011",1.7526,73.935496,€95.6M
179844,Diego Costa,29,Spain,85,85,Atlético de Madrid,38.5,100000,LS,"Jan 1, 2018",1.8796,86.182480,€78.9M
164169,Filipe Luís,32,Brazil,85,85,Atlético de Madrid,21.5,81000,LB,"Jul 28, 2015",1.8288,73.935496,€44.1M
216460,J. Giménez,23,Uruguay,84,88,Atlético de Madrid,34.0,63000,RCB,"Jul 1, 2013",1.8542,79.832192,€72.3M
213565,T. Lemar,22,France,83,89,Atlético de Madrid,35.5,64000,LM,"Jul 1, 2018",1.7018,62.142104,€79.9M


### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 2</font> 

Mostrar las filas correspondientes a los jugadores españoles cuyo club es `Atlético de Madrid`. Utilizar selección condicional.

In [12]:
df_fifa[(df_fifa['Club'] == 'Atlético de Madrid') & (df_fifa['Nationality'] == 'Spain')]

,Name,Age,Nationality,Overall,Potential,Club,Value,Wage,Position,Joined,Height,Weight,Release Clause
ID,,,,,,,,,,,,,
208421,Saúl,23,Spain,85,90,Atlético de Madrid,50.5,77000,RCM,"Jul 1, 2013",1.8288,77.110640,€107.3M
193747,Koke,26,Spain,85,86,Atlético de Madrid,45.0,88000,LM,"Jan 1, 2011",1.7526,73.935496,€95.6M
179844,Diego Costa,29,Spain,85,85,Atlético de Madrid,38.5,100000,LS,"Jan 1, 2018",1.8796,86.182480,€78.9M
178086,Adán,31,Spain,83,83,Atlético de Madrid,15.5,55000,GK,"Jul 10, 2018",1.9050,92.079176,€31.8M
231866,Rodri,22,Spain,82,87,Atlético de Madrid,29.0,59000,CM,"Jul 1, 2018",1.9050,82.100152,€61.6M
199715,Vitolo,28,Spain,81,81,Atlético de Madrid,18.0,65000,LM,"Jul 12, 2017",1.8542,78.925008,€36.9M
146760,Juanfran,33,Spain,80,80,Atlético de Madrid,5.5,53000,RB,"Jan 1, 2011",1.8034,71.213944,€11.3M
229865,Óscar Pinchi,22,Spain,66,77,Atlético de Madrid,1.1,11000,LM,"Jul 2, 2018",1.7272,62.142104,€2.5M
243652,Borja Garcés,18,Spain,64,83,Atlético de Madrid,950.0,5000,ST,"Jul 1, 2018",1.7780,71.213944,€2.5M


### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 3</font> 

Repetir el ejercicio anterior, pero utilizando el método `DataFrame.query()`.

In [13]:
df_fifa.query('Club == "Atlético de Madrid" and Nationality == "Spain"')

,Name,Age,Nationality,Overall,Potential,Club,Value,Wage,Position,Joined,Height,Weight,Release Clause
ID,,,,,,,,,,,,,
208421,Saúl,23,Spain,85,90,Atlético de Madrid,50.5,77000,RCM,"Jul 1, 2013",1.8288,77.110640,€107.3M
193747,Koke,26,Spain,85,86,Atlético de Madrid,45.0,88000,LM,"Jan 1, 2011",1.7526,73.935496,€95.6M
179844,Diego Costa,29,Spain,85,85,Atlético de Madrid,38.5,100000,LS,"Jan 1, 2018",1.8796,86.182480,€78.9M
178086,Adán,31,Spain,83,83,Atlético de Madrid,15.5,55000,GK,"Jul 10, 2018",1.9050,92.079176,€31.8M
231866,Rodri,22,Spain,82,87,Atlético de Madrid,29.0,59000,CM,"Jul 1, 2018",1.9050,82.100152,€61.6M
199715,Vitolo,28,Spain,81,81,Atlético de Madrid,18.0,65000,LM,"Jul 12, 2017",1.8542,78.925008,€36.9M
146760,Juanfran,33,Spain,80,80,Atlético de Madrid,5.5,53000,RB,"Jan 1, 2011",1.8034,71.213944,€11.3M
229865,Óscar Pinchi,22,Spain,66,77,Atlético de Madrid,1.1,11000,LM,"Jul 2, 2018",1.7272,62.142104,€2.5M
243652,Borja Garcés,18,Spain,64,83,Atlético de Madrid,950.0,5000,ST,"Jul 1, 2018",1.7780,71.213944,€2.5M


### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 4</font> 

Mostrar los juguadores españoles cuyo club es el Atlético de Madrid, y para los que la diferencia entre su potencial (`Potential`) y valoración global (`Overall`) es menor que cinco.

In [14]:
df_fifa[df_fifa['Potential']-df_fifa['Overall'] < 5].query('Club == "Atlético de Madrid" and Nationality == "Spain"')

,Name,Age,Nationality,Overall,Potential,Club,Value,Wage,Position,Joined,Height,Weight,Release Clause
ID,,,,,,,,,,,,,
193747,Koke,26,Spain,85,86,Atlético de Madrid,45.0,88000,LM,"Jan 1, 2011",1.7526,73.935496,€95.6M
179844,Diego Costa,29,Spain,85,85,Atlético de Madrid,38.5,100000,LS,"Jan 1, 2018",1.8796,86.182480,€78.9M
178086,Adán,31,Spain,83,83,Atlético de Madrid,15.5,55000,GK,"Jul 10, 2018",1.9050,92.079176,€31.8M
199715,Vitolo,28,Spain,81,81,Atlético de Madrid,18.0,65000,LM,"Jul 12, 2017",1.8542,78.925008,€36.9M
146760,Juanfran,33,Spain,80,80,Atlético de Madrid,5.5,53000,RB,"Jan 1, 2011",1.8034,71.213944,€11.3M


### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 5</font> 

Mostrar los jugadores que militan en el Real Madrid, FC Barcelona o Atlético de Madrid. Utilizar el método `Series.isin()`.

In [15]:
df_fifa[["Name", "Club"]][df_fifa["Club"].isin(['Atlético de Madrid', 'Real Madrid', 'FC Barcelona'])]

,Name,Club
ID,,
158023,L. Messi,FC Barcelona
177003,L. Modrić,Real Madrid
176580,L. Suárez,FC Barcelona
155862,Sergio Ramos,Real Madrid
200389,J. Oblak,Atlético de Madrid
...,...,...
246139,Iñaki Peña,FC Barcelona
236940,Darío Poveda,Atlético de Madrid
245986,Cristian Rodríguez,Atlético de Madrid


### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 6</font> 

Mostrar los jugadores cuya altura está entre 1.70 y 1.90 cm. Utilizar  el método `Series.between()`.

In [16]:
df_fifa[["Name", "Height"]][df_fifa["Height"].between(1.70, 1.90)]

,Name,Height
ID,,
158023,L. Messi,1.7018
20801,Cristiano Ronaldo,1.8796
190871,Neymar Jr,1.7526
192985,K. De Bruyne,1.8034
183277,E. Hazard,1.7272
...,...,...
243413,D. Walsh,1.8542
238813,J. Lundstram,1.7526
241638,B. Worman,1.7272




<a id="section92"></a> 
### <font color="#5E5E5E">9.2 Filtrado por valor del índice</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 7</font> 

El *DataFrame* `df_fifa_att` contiene columnas con información sobre las características de los jugadores. Las características de los porteros se distinguen porque contienen las letras `GK` en su nombre. Mostrar las columnas correspondientes a estas características.

In [17]:
df_fifa_att.filter(regex='^GK')

,GKDiving,GKHandling,GKKicking,GKPositioning,GKReflexes
ID,,,,,
158023,6.0,11.0,15.0,14.0,8.0
20801,7.0,11.0,15.0,14.0,11.0
190871,9.0,9.0,15.0,15.0,11.0
193080,90.0,85.0,87.0,88.0,94.0
192985,15.0,13.0,5.0,10.0,13.0
...,...,...,...,...,...
238813,10.0,13.0,7.0,8.0,9.0
243165,10.0,9.0,9.0,5.0,12.0
241638,6.0,5.0,10.0,6.0,13.0



<a id="section93"></a> 
### <font color="#5E5E5E">9.3 Reemplazo condicional de valores</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 8</font> 

Mostrar las filas correspondientes a los jugadores del `FC Barcelona` y sustituir los valores del resto con la cadena de texto ` '-----'`.

In [ ]:
df_fifa_copy = df_fifa
df_fifa_copy.loc[df_fifa_copy["Club"] != "FC Barcelona", "Club"] = "-----"
df_fifa_copy


,ID,Name,Age,Nationality,Overall,Potential,Club,Value,Wage,Position,Joined,Height,Weight,Release Clause
0,158023,L. Messi,31,Argentina,94,94,FC Barcelona,110.5,565000,RF,"Jul 1, 2004",1.7018,72.121128,€226.5M
1,20801,Cristiano Ronaldo,33,Portugal,94,94,-----,77.0,405000,ST,"Jul 10, 2018",1.8796,83.007336,€127.1M
2,190871,Neymar Jr,26,Brazil,92,93,-----,118.5,290000,LW,"Aug 3, 2017",1.7526,68.038800,€228.1M
3,193080,De Gea,27,Spain,91,93,-----,72.0,260000,GK,"Jul 1, 2011",1.9304,76.203456,€138.6M
4,192985,K. De Bruyne,27,Belgium,91,92,-----,102.0,355000,RCM,"Aug 30, 2015",1.8034,69.853168,€196.4M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16638,238813,J. Lundstram,19,England,47,65,-----,60.0,1000,CM,"May 3, 2017",1.7526,60.781328,€143K
16639,243165,N. Christoffersson,19,Sweden,47,63,-----,60.0,1000,ST,"Mar 19, 2018",1.9050,77.110640,€113K
16640,241638,B. Worman,16,England,47,67,-----,60.0,1000,ST,"Jul 1, 2017",1.7272,67.131616,€165K
16641,246268,D. Walker-Rice,17,England,47,66,-----,60.0,1000,RW,"Apr 24, 2018",1.7780,69.853168,€143K


<div align="right">
<a href="#indice"><font size=5 color=#B51700><i class="fa fa-arrow-circle-up" aria-hidden="true" ></i></font></a>
</div>

---

<a id="section10"></a>
## <font color="#5E5E5E" size=5> 10. Agrupaciones </font>


<a id="section101"></a> 
### <font color="#5E5E5E">10.1 Agrupación por categorías mediante `DataFrame.groupby()` </font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 9</font> 

Mostrar el número de jugadores de cada nacionalidad ordenado de mayor a menor.

In [40]:
df_fifa.groupby("Nationality").size().sort_values(ascending=False)

Nationality
England                 1475
Germany                 1151
Spain                    974
France                   853
Argentina                833
                        ... 
Oman                       1
Palestine                  1
South Sudan                1
São Tomé & Príncipe        1
United Arab Emirates       1
Length: 161, dtype: int64

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 10</font>

Agrupar los jugadores por nacionalidad y posición, y ordenar los grupos por salario medio de forma descendiente.

In [65]:
df_fifa_copy = df_fifa.copy()

df_fifa_copy.groupby(["Nationality", "Position", "Name"])["Wage"].mean().sort_values(ascending=False)

Nationality  Position  Name             
Argentina    RF        L. Messi             565000.0
Uruguay      RS        L. Suárez            455000.0
Croatia      RCM       L. Modrić            420000.0
Portugal     ST        Cristiano Ronaldo    405000.0
Spain        RCB       Sergio Ramos         380000.0
                                              ...   
England      ST        C. Andrew              1000.0
Australia    GK        M. Sutton              1000.0
                       N. James               1000.0
                       N. Suman               1000.0
England      ST        A. Oteh                1000.0
Name: Wage, Length: 16598, dtype: float64

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 11</font> 

Repetir el ejercicio anterior, pero de modo que la nacionalidad y la posición sean columnas en el *DataFrame* resultante.

In [67]:
df_fifa_copy = df_fifa.copy()

df_fifa_copy.groupby(["Nationality", "Position", "Name"])["Wage"].mean().sort_values(ascending=False).reset_index()

,Nationality,Position,Name,Wage
0,Argentina,RF,L. Messi,565000.0
1,Uruguay,RS,L. Suárez,455000.0
2,Croatia,RCM,L. Modrić,420000.0
3,Portugal,ST,Cristiano Ronaldo,405000.0
4,Spain,RCB,Sergio Ramos,380000.0
...,...,...,...,...
16593,England,ST,C. Andrew,1000.0
16594,Australia,GK,M. Sutton,1000.0
16595,Australia,GK,N. James,1000.0
16596,Australia,GK,N. Suman,1000.0


### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 12</font> 

Mostrar el valor medio de los jugadores cuya valoración media está por encima de 70 y el de los que están por debajo.

In [116]:
df_fifa_copy = df_fifa.copy()
x = df_fifa_copy.groupby(df_fifa["Overall"] > 70)["Value"].mean()
y = df_fifa_copy.groupby(df_fifa_copy["Overall"] > 70)["Value"].mean()
print(x)
print(y)

Overall
False    352.826876
True      28.549606
Name: Value, dtype: float64
Overall
False    352.826876
True      28.549606
Name: Value, dtype: float64


### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 13</font> 

Mostrar la media de todas las columnas para los jugadores cuya valoración media está por encima de 70 y el de los que están por debajo.

In [105]:
df_fifa_copy_2 = df_fifa.copy()
media1 = df_fifa_copy_2.groupby(df_fifa_copy_2["Overall"] > 70).mean(numeric_only=True)
media2 = df_fifa_copy_2.groupby(df_fifa_copy_2["Overall"] < 70).mean(numeric_only=True)
print(media1)
print(media2)

                    ID        Age    Overall  Potential       Value  \
Overall                                                               
False    219054.813661  24.461588  63.092074  69.046159  352.826876   
True     198965.202734  27.410102  74.933040  77.123726   28.549606   

                 Wage    Height     Weight  
Overall                                     
False     3137.665288  1.811395  74.875503  
True     28126.737720  1.815564  76.475653  
                    ID        Age    Overall  Potential       Value  \
Overall                                                               
False    199753.125684  27.358483  74.161650  76.423964   32.952795   
True     220099.376421  24.279865  62.612649  68.796131  373.378199   

                 Wage    Height     Weight  
Overall                                     
False    24995.308835  1.814943  76.337689  
True      2793.181227  1.811381  74.825681  


---

<a id="section107"></a> 
### <font color="#5E5E5E">10.7 Agrupación por categorías externas. Grupos a partir de datos numéricos.</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 14</font> 

Crear categorías de jugador según su valoración media (`Overall`) esté: en el 25% más bajo (`Muy malo`); entre el 25% y el 50% (`Malo`); entre el 50% y el 75% (`Medio`); entre el 75% y el 95% más alto (`Bueno`); y en el 5% más alto (`Top`). Mostrar el número de jugadores y el valor (`Overall`) medio que caen dentro de cada categoría.

In [90]:
media_jugadores = df_fifa[["Name","Overall"]].groupby("Name").mean(numeric_only=True).sort_values(by="Overall")


limites = [
    media_jugadores["Overall"].min() - 1,  
    np.percentile(media_jugadores["Overall"], 25),
    np.percentile(media_jugadores["Overall"], 50),
    np.percentile(media_jugadores["Overall"], 75),
    np.percentile(media_jugadores["Overall"], 95),
    media_jugadores["Overall"].max() + 1 
]

notas = ['Muy malo', 'Malo', 'Medio', 'Bueno', 'Top']


media_jugadores['Rendimiento'] = pd.cut(
    media_jugadores['Overall'], 
    bins=limites, 
    labels=notas, 
    ordered=True
)


df_final = media_jugadores.rename(columns={"Overall": "Puntuacion"}).sort_values(by='Rendimiento', ascending=False)
df_final 



,Puntuacion,Rendimiento
Name,,
Cristiano Ronaldo,94.0,Top
J. Lerma,79.0,Top
Raphinha,79.0,Top
R. Centurión,79.0,Top
P. Wernbloom,79.0,Top
...,...,...
T. Siekman,60.0,Muy malo
E. Katayama,60.0,Muy malo
J. Kamata,60.0,Muy malo


<div align="right">
<a href="#indice"><font size=5 color=#B51700><i class="fa fa-arrow-circle-up" aria-hidden="true" ></i></font></a>
</div>

---

<a id="section11"></a> 
## <font color="#5E5E5E">11 Operaciones sobre grupos</font>

<a id="section111"></a> 
### <font color="#5E5E5E">11.1 Agregación en grupos.</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 15</font> 

Mostrar los valores máximo y mínimo de `Overall` y `Wage` para cada posición.

In [100]:
df_fifa.groupby("Position")[["Overall","Wage"]].max(numeric_only=True).sort_values(by="Position")

,Overall,Wage
Position,,
CAM,89,205000
CB,90,205000
CDM,89,315000
CF,82,71000
CM,86,175000
GK,91,260000
LAM,81,31000
LB,88,285000
LCB,89,215000




<a id="section112"></a> 
### <font color="#5E5E5E">11.2 Filtrado por grupos</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 16</font> 

Crear un *DataFrame* con las filas correspondientes a los clubs en los que el salario (`Wage`) medio está por encima de la media de los salarios.

In [134]:
media_de_salarios = df_fifa["Wage"].mean()
media_de_salarios_de_cada_club = df_fifa.groupby("Club")["Wage"].mean()
dat = pd.DataFrame(media_de_salarios_de_cada_club)
dat[dat["Wage"]>media_de_salarios].sort_values(by="Wage", ascending=False)

,Wage
Club,
Real Madrid,152030.303030
FC Barcelona,146575.757576
Juventus,131680.000000
Manchester City,113363.636364
Manchester United,102757.575758
...,...
Aston Villa,9920.000000
FC Ingolstadt 04,9866.666667
Standard de Liège,9680.000000


### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 17</font> 

Filtrar los datos de modo que se conserven solamente los jugadores que pertenezcan a clubes donde la valoración media es mayor que 75, y que pertenezcan a países para los que hay más de 10 jugadores en estos clubs. Almacenar el resultado en `df_fifa`.

In [159]:
valoracion_media = df_fifa["Overall"].mean()
media_de_valoraciones_de_cada_club = df_fifa.groupby("Club")["Overall"].mean()
dat = pd.DataFrame(media_de_valoraciones_de_cada_club)
num_jugadores_por_club = df_fifa.groupby("Club")["Name"].count()
dat_j = pd.DataFrame(num_jugadores_por_club)
club_mas_de_diez = dat_j[dat_j["Name"]>10]
dat["Numero de jugadores"]=club_mas_de_diez
dat.sort_values(by="Numero de jugadores", ascending=False)

,Overall,Numero de jugadores
Club,,
Arsenal,75.181818,33
AS Monaco,72.939394,33
Atlético de Madrid,75.060606,33
Liverpool,76.000000,33
FC Barcelona,78.030303,33
...,...,...
CF Rayo Majadahonda,66.500000,18
Defensa y Justicia,67.666667,18
Perugia,65.000000,17



<a id="section113"></a> 
### <font color="#5E5E5E">11.3 Transformación por grupo</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 18</font> 

Mostar, para cada jugador (`ID`) qué porcentaje supone su salario sobre el total de salarios (`Wage`) de su club.

In [67]:
salarios_de_clubs_total = df_fifa.groupby("Club")["Wage"].sum()
salarios_de_jugadores = df_fifa.groupby("Club")[["Name", "Wage"]]
salarios_j = salarios_de_jugadores.apply(lambda x: x)
salarios = pd.DataFrame({"Salarios":salarios_de_clubs_total})
salarios_j["Porcentaje salario"] = salarios_j["Wage"]*100/salarios["Salarios"]
salarios_j["Total salarios club"] = salarios["Salarios"]
salarios_j.sort_values(by="Wage", ascending=False)

Name    Wage  Porcentaje salario  \
Club          ID                                                      
FC Barcelona  158023           L. Messi  565000           11.680794   
              176580          L. Suárez  455000            9.406657   
Real Madrid   177003          L. Modrić  420000            8.371537   
Juventus      20801   Cristiano Ronaldo  405000           12.302552   
Real Madrid   155862       Sergio Ramos  380000            7.574248   
...                                 ...     ...                 ...   
Śląsk Wrocław 243921         J. Łabojko    1000            2.000000   
              244512        D. Szczepan    1000            2.000000   
              243922         M. Radecki    1000            2.000000   
              219701          J. Wrąbel    1000            2.000000   
              244532      P. Kucharczyk    1000            2.000000   

                      Total salarios club  
Club          ID                           
FC Barcelona  158023              4837000  
              176580              4837000  
Real Madrid   177003              5017000  
Juventus      20801               3292000  
Real Madrid   155862              5017000  
...                                   ...  
Śląsk Wrocław 243921                50000  
              244512                50000  
              243922                50000  
              219701                50000  
              244532                50000  

[16643 rows x 4 columns]

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 19</font> 

Repetir el ejercicio anterior, pero utilizar `DataFrame.assign()` para añadir el resultado a una columna denominada `df_fifa['value_club']`. Hacer el ejercicio también sin utilizar `DataFrame.assign()`.

In [ ]:
#CON ASSIGN
salarios_de_clubs_total = df_fifa.groupby("Club")["Wage"].sum()
salarios_j = df_fifa.groupby("Club")[["Name", "Wage"]].apply(lambda x: x)
salarios_j = salarios_j.assign(**{"Total salarios club": salarios_j.index.get_level_values("Club").map(salarios_de_clubs_total)})
salarios_j = salarios_j.assign(**{"Porcentaje salario": salarios_j["Wage"] * 100 / salarios_j["Total salarios club"]})
salarios_j.sort_values(by="Wage", ascending=False)

#SIN ASSIGN
salarios_de_clubs_total_2 = df_fifa.groupby("Club")["Wage"].sum()
salarios_j_2 = df_fifa.groupby("Club")[["Name", "Wage"]].apply(lambda x: x)
salarios_j_2["Total salarios club"] = salarios_j_2.index.get_level_values("Club").map(salarios_de_clubs_total_2)
salarios_j_2["Porcentaje salario"] = salarios_j_2["Wage"] * 100 / salarios_j["Total salarios club"]
salarios_j_2.sort_values(by="Wage", ascending=False)

Name    Wage  Total salarios club  \
Club          ID                                                       
FC Barcelona  158023           L. Messi  565000              4837000   
              176580          L. Suárez  455000              4837000   
Real Madrid   177003          L. Modrić  420000              5017000   
Juventus      20801   Cristiano Ronaldo  405000              3292000   
Real Madrid   155862       Sergio Ramos  380000              5017000   
...                                 ...     ...                  ...   
Śląsk Wrocław 243921         J. Łabojko    1000                50000   
              244512        D. Szczepan    1000                50000   
              243922         M. Radecki    1000                50000   
              219701          J. Wrąbel    1000                50000   
              244532      P. Kucharczyk    1000                50000   

                      Porcentaje salario  
Club          ID                          
FC Barcelona  158023           11.680794  
              176580            9.406657  
Real Madrid   177003            8.371537  
Juventus      20801            12.302552  
Real Madrid   155862            7.574248  
...                                  ...  
Śląsk Wrocław 243921            2.000000  
              244512            2.000000  
              243922            2.000000  
              219701            2.000000  
              244532            2.000000  

[16643 rows x 4 columns]

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 20</font> 

Repetir el ejercicio anterior, ordenando el resultado de manera descendente según el valor de la nueva columna, y mostrando solamente los jugadores cuya nacionalidad es española.

In [78]:
salarios_j_2["Nacionalidad"] = df_fifa.groupby("Club")[["Nationality"]].apply(lambda x: x)
salarios_j_2[salarios_j_2["Nacionalidad"] == "Spain"].sort_values(by="Porcentaje salario", ascending=False)

Name   Wage  Total salarios club  \
Club                ID                                                    
Vissel Kobe         41              Iniesta  21000                80000   
Sagan Tosu          49369   Fernando Torres  10000                63000   
Górnik Zabrze       138306           Angulo   6000                38000   
Willem II           210441         Fran Sol  11000                72000   
Guizhou Hengfeng FC 171018     Mario Suárez  16000               112000   
...                                     ...    ...                  ...   
Real Madrid         243155    Manu Hernando   9000              5017000   
                    245287     Sergio López   9000              5017000   
Manchester City     245037      Eric García   6000              3741000   
Valencia CF         246281    Emilio Bernad   1000               785000   
FC Barcelona        246139       Iñaki Peña   4000              4837000   

                            Porcentaje salario Nacionalidad  
Club                ID                                       
Vissel Kobe         41               26.250000        Spain  
Sagan Tosu          49369            15.873016        Spain  
Górnik Zabrze       138306           15.789474        Spain  
Willem II           210441           15.277778        Spain  
Guizhou Hengfeng FC 171018           14.285714        Spain  
...                                        ...          ...  
Real Madrid         243155            0.179390        Spain  
                    245287            0.179390        Spain  
Manchester City     245037            0.160385        Spain  
Valencia CF         246281            0.127389        Spain  
FC Barcelona        246139            0.082696        Spain  

[974 rows x 5 columns]



<a id="section114"></a> 
### <font color="#5E5E5E"> 11.4 `DataFrameGroupBy.apply()`</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 21</font> 

Mostrar, para cada club, el valor del jugador con mayor valoración. Ordenar el resultado de manera descendiente. Repetir el ejercicio pero utilizando `DataFrameGroupBy.apply()`.

In [87]:
#SIN APPLY
valoracion_clubs_max = df_fifa.groupby("Club")[["Overall", "Name"]].max().sort_values(by="Overall",ascending=False)
valoracion_clubs_max
#CON APPLY
valoracion_clubs_max = df_fifa.groupby("Club")[["Overall", "Name"]].apply(lambda x: x.max()).sort_values(by="Overall",ascending=False)
valoracion_clubs_max


,Overall,Name
Club,,
Juventus,94,W. Szczęsny
FC Barcelona,94,T. Vermaelen
Paris Saint-Germain,92,Y. Adli
Real Madrid,91,Vinícius Júnior
Manchester City,91,V. Kompany
...,...,...
Bohemian FC,64,S. Bohan
Sligo Rovers,63,R. Murray
Limerick FC,62,W. Fitzgerald


### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 22</font> 

Repetir el ejercicio anterior, pero mostrando la fila completa correspondiente al jugador.

In [ ]:
#SIN APPLY
valoracion_clubs_max = df_fifa.groupby("Club").max().sort_values(by="Overall",ascending=False)
valoracion_clubs_max
#CON APPLY
valoracion_clubs_max = df_fifa.groupby("Club").apply(lambda x: x.max()).sort_values(by="Overall",ascending=False)
valoracion_clubs_max

,Name,Age,Nationality,Overall,Potential,Value,Wage,Position,Joined,Height,Weight,Release Clause
Club,,,,,,,,,,,,
Juventus,W. Szczęsny,37,Uruguay,94,94,575.0,405000,ST,"Jul 24, 2017",1.9558,93.893544,€76.7M
FC Barcelona,T. Vermaelen,32,Uruguay,94,94,975.0,565000,ST,"Sep 1, 2008",1.9304,86.182480,€95.3M
Paris Saint-Germain,Y. Adli,40,Uruguay,92,95,875.0,290000,ST,"Jul 6, 2018",1.9558,93.893544,€9.1M
Real Madrid,Vinícius Júnior,32,Wales,91,92,875.0,420000,ST,"Sep 2, 2013",1.9812,96.161504,€88.2M
Manchester City,V. Kompany,35,Ukraine,91,92,600.0,355000,ST,"Oct 7, 2016",1.9812,86.182480,€990K
...,...,...,...,...,...,...,...,...,...,...,...,...
Bohemian FC,S. Bohan,33,Romania,64,72,475.0,1000,ST,"Sep 8, 2018",1.8796,86.182480,€93K
Sligo Rovers,R. Murray,36,Scotland,63,72,450.0,1000,ST,"May 25, 2018",1.9812,82.100152,€731K
Limerick FC,W. Fitzgerald,31,Republic of Ireland,62,70,300.0,1000,ST,"Oct 27, 2017",1.9558,83.007336,€88K


### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 23</font> 

Repetir el ejercicio anterior pero mostrando, para cada equipo, el jugador con mayor y menor valoración media.

In [118]:
#SIN APPLY
valoracion_clubs_max = df_fifa.groupby("Club")["Overall"].idxmax()
valoracion_clubs_min= df_fifa.groupby("Club")["Overall"].idxmin()
#CON APPLY
valoracion_clubs_max = df_fifa.groupby("Club")["Overall"].apply(lambda x: x.idxmax())
valoracion_clubs_min = df_fifa.groupby("Club")["Overall"].apply(lambda x: x.idxmin())
dat = pd.DataFrame({"Jugador max": df_fifa.loc[valoracion_clubs_max, "Name"].values,"Valoracion max": df_fifa.loc[valoracion_clubs_max, "Overall"].values,"Jugador min": df_fifa.loc[valoracion_clubs_min, "Name"].values,"Valoracion min": df_fifa.loc[valoracion_clubs_min, "Overall"].values}, index=valoracion_clubs_max.index)
dat


,Jugador max,Valoracion max,Jugador min,Valoracion min
Club,,,,
SSV Jahn Regensburg,M. Grüttner,71,A. Weidinger,52
1. FC Heidenheim 1846,M. Schnatterer,75,J. Brändle,53
1. FC Kaiserslautern,L. Spalvis,70,J. Scholz,54
1. FC Köln,T. Horn,83,J. Bartels,53
1. FC Magdeburg,A. Ignjovski,70,P. Harant,57
...,...,...,...,...
Zagłębie Sosnowiec,K. Wrzesiński,65,J. Pawlik,51
Çaykur Rizespor,A. Umar,76,O. Malcı,56
Örebro SK,N. Besara,71,N. Ayéva,48


<div align="right">
<a href="#indice"><font size=5 color=#B51700><i class="fa fa-arrow-circle-up" aria-hidden="true" ></i></font></a>
</div>

---

<a id="section13"></a>
## <font color="#5E5E5E"> 13. Combinación de *Series* y *DataFrames* </font>



<a id="section132"></a> 
### <font color="#5E5E5E">13.2 Concatenación de *Series* y *DataFrames*</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 24</font> 

Concatenar las columnas de los *DataFrame* `df_fifa` y `df_fifa_att` de modo que el *DataFrame* resutante contenga las 200 filas correspondientes a los jugadores que aparecen en `df_fifa`. Utilizar para ello `pd.concat()`.

In [126]:
df_fifa_total = pd.concat([df_fifa, df_fifa_att], axis=1)
df_fifa_total

,Name,Age,Nationality,Overall,Potential,Club,Value,Wage,Position,Joined,Height,Weight,Release Clause
ID,,,,,,,,,,,,,
158023,L. Messi,31,Argentina,94,94,FC Barcelona,110.5,565000,RF,"Jul 1, 2004",1.7018,72.121128,€226.5M
20801,Cristiano Ronaldo,33,Portugal,94,94,Juventus,77.0,405000,ST,"Jul 10, 2018",1.8796,83.007336,€127.1M
190871,Neymar Jr,26,Brazil,92,93,Paris Saint-Germain,118.5,290000,LW,"Aug 3, 2017",1.7526,68.038800,€228.1M
193080,De Gea,27,Spain,91,93,Manchester United,72.0,260000,GK,"Jul 1, 2011",1.9304,76.203456,€138.6M
192985,K. De Bruyne,27,Belgium,91,92,Manchester City,102.0,355000,RCM,"Aug 30, 2015",1.8034,69.853168,€196.4M
...,...,...,...,...,...,...,...,...,...,...,...,...,...
238813,J. Lundstram,19,England,47,65,Crewe Alexandra,60.0,1000,CM,"May 3, 2017",1.7526,60.781328,€143K
243165,N. Christoffersson,19,Sweden,47,63,Trelleborgs FF,60.0,1000,ST,"Mar 19, 2018",1.9050,77.110640,€113K
241638,B. Worman,16,England,47,67,Cambridge United,60.0,1000,ST,"Jul 1, 2017",1.7272,67.131616,€165K




<a id="section133"></a> 
### <font color="#5E5E5E">13.3 Fusión de  *DataFrames*</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 25</font> 

Fundir las filas de los *DataFrames* `df_fifa` y `df_fifa_att` utilizando como clave de unión el índice y de modo que el *DataFrame* resultante contenga los índices del *DataFrame* `df_fifa`. Utilizar ahora `DataFrame.merge()`. Almacenar el resultado en `df_fifa`.

In [128]:
df_fifa_total = df_fifa.merge(df_fifa_att,left_index=True,right_index=True,how="left")

df_fifa_total

,Name,Age,Nationality,Overall,Potential,Club,Value,Wage,Position,Joined,Height,Weight,Release Clause
ID,,,,,,,,,,,,,
158023,L. Messi,31,Argentina,94,94,FC Barcelona,110.5,565000,RF,"Jul 1, 2004",1.7018,72.121128,€226.5M
20801,Cristiano Ronaldo,33,Portugal,94,94,Juventus,77.0,405000,ST,"Jul 10, 2018",1.8796,83.007336,€127.1M
190871,Neymar Jr,26,Brazil,92,93,Paris Saint-Germain,118.5,290000,LW,"Aug 3, 2017",1.7526,68.038800,€228.1M
193080,De Gea,27,Spain,91,93,Manchester United,72.0,260000,GK,"Jul 1, 2011",1.9304,76.203456,€138.6M
192985,K. De Bruyne,27,Belgium,91,92,Manchester City,102.0,355000,RCM,"Aug 30, 2015",1.8034,69.853168,€196.4M
...,...,...,...,...,...,...,...,...,...,...,...,...,...
238813,J. Lundstram,19,England,47,65,Crewe Alexandra,60.0,1000,CM,"May 3, 2017",1.7526,60.781328,€143K
243165,N. Christoffersson,19,Sweden,47,63,Trelleborgs FF,60.0,1000,ST,"Mar 19, 2018",1.9050,77.110640,€113K
241638,B. Worman,16,England,47,67,Cambridge United,60.0,1000,ST,"Jul 1, 2017",1.7272,67.131616,€165K


<div align="right">
<a href="#indice"><font size=5 color=#B51700><i class="fa fa-arrow-circle-up" aria-hidden="true" ></i></font></a>
</div>

---

<a id="section14"></a>
## <font color="#5E5E5E"> 14. Índices de varios niveles</font>


### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 26</font> 

Establecer como índice principal de `df_fifa` la columna `Club` y como secundario el `ID` del jugador. Para ello, hay que resetear perimero el índice. 

In [149]:
df_fifa = df_fifa.set_index(["Club"])
df_fifa

,level_0,index,Name,Age,Photo,Nationality,Flag,Overall,Potential,Club Logo,...,Composure,Marking,StandingTackle,SlidingTackle,GKDiving,GKHandling,GKKicking,GKPositioning,GKReflexes,Release Clause
Club,,,,,,,,,,,,,,,,,,,,,
FC Barcelona,0,0,L. Messi,31,https://cdn.sofifa.org/players/4/19/158023.png,Argentina,https://cdn.sofifa.org/flags/52.png,94,94,https://cdn.sofifa.org/teams/2/light/241.png,...,96.0,33.0,28.0,26.0,6.0,11.0,15.0,14.0,8.0,€226.5M
Juventus,1,1,Cristiano Ronaldo,33,https://cdn.sofifa.org/players/4/19/20801.png,Portugal,https://cdn.sofifa.org/flags/38.png,94,94,https://cdn.sofifa.org/teams/2/light/45.png,...,95.0,28.0,31.0,23.0,7.0,11.0,15.0,14.0,11.0,€127.1M
Paris Saint-Germain,2,2,Neymar Jr,26,https://cdn.sofifa.org/players/4/19/190871.png,Brazil,https://cdn.sofifa.org/flags/54.png,92,93,https://cdn.sofifa.org/teams/2/light/73.png,...,94.0,27.0,24.0,33.0,9.0,9.0,15.0,15.0,11.0,€228.1M
Manchester United,3,3,De Gea,27,https://cdn.sofifa.org/players/4/19/193080.png,Spain,https://cdn.sofifa.org/flags/45.png,91,93,https://cdn.sofifa.org/teams/2/light/11.png,...,68.0,15.0,21.0,13.0,90.0,85.0,87.0,88.0,94.0,€138.6M
Manchester City,4,4,K. De Bruyne,27,https://cdn.sofifa.org/players/4/19/192985.png,Belgium,https://cdn.sofifa.org/flags/7.png,91,92,https://cdn.sofifa.org/teams/2/light/10.png,...,88.0,68.0,58.0,51.0,15.0,13.0,5.0,10.0,13.0,€196.4M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Crewe Alexandra,18202,18202,J. Lundstram,19,https://cdn.sofifa.org/players/4/19/238813.png,England,https://cdn.sofifa.org/flags/14.png,47,65,https://cdn.sofifa.org/teams/2/light/121.png,...,45.0,40.0,48.0,47.0,10.0,13.0,7.0,8.0,9.0,€143K
Trelleborgs FF,18203,18203,N. Christoffersson,19,https://cdn.sofifa.org/players/4/19/243165.png,Sweden,https://cdn.sofifa.org/flags/46.png,47,63,https://cdn.sofifa.org/teams/2/light/703.png,...,42.0,22.0,15.0,19.0,10.0,9.0,9.0,5.0,12.0,€113K
Cambridge United,18204,18204,B. Worman,16,https://cdn.sofifa.org/players/4/19/241638.png,England,https://cdn.sofifa.org/flags/14.png,47,67,https://cdn.sofifa.org/teams/2/light/1944.png,...,41.0,32.0,13.0,11.0,6.0,5.0,10.0,6.0,13.0,€165K


### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 27</font> 

Acceder a los jugadores del Liverpool y al jugador `203376` de ese equipo.

In [152]:
df_fifa.loc["Liverpool", "Name"]


Club
Liverpool                 M. Salah
Liverpool                  S. Mané
Liverpool              V. van Dijk
Liverpool          Roberto Firmino
Liverpool                  Alisson
Liverpool                  Fabinho
Liverpool                 N. Keïta
Liverpool             A. Robertson
Liverpool             J. Henderson
Liverpool             G. Wijnaldum
Liverpool                D. Lovren
Liverpool               X. Shaqiri
Liverpool                J. Milner
Liverpool    A. Oxlade-Chamberlain
Liverpool                 J. Matip
Liverpool               A. Lallana
Liverpool             D. Sturridge
Liverpool                 N. Clyne
Liverpool              S. Mignolet
Liverpool      T. Alexander-Arnold
Liverpool                 J. Gomez
Liverpool                 D. Origi
Liverpool           Alberto Moreno
Liverpool              L. Marković
Liverpool               D. Solanke
Liverpool         Pedro Chirivella
Liverpool              R. Brewster
Liverpool                 C. Jones
Liverpool      

<div align="right">
<a href="#indice"><font size=5 color=#B51700><i class="fa fa-arrow-circle-up" aria-hidden="true" ></i></font></a>
</div>

---

<a id="section15"></a>
## <font color="#5E5E5E"> 15. Tablas dinámicas y reestructuración </font>

<a id="section151"></a> 
### <font color="#5E5E5E">15.1 Creación de tablas dinámicas</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 28</font> 

Crear una tabla pivot en la que aparezca la media de valoración y salario por equipo y nacionalidad.

In [159]:
df_fifa["Overall"] = pd.to_numeric(df_fifa["Overall"], errors="coerce")
df_fifa["Wage"] = pd.to_numeric(df_fifa["Wage"], errors="coerce")
df_fifa.pivot_table(
    values=["Overall", "Wage"],
    index=["Club", "Nationality"],
    aggfunc="mean"
)

Overall
Club                 Nationality           
 SSV Jahn Regensburg Armenia      70.000000
                     Denmark      66.000000
                     Germany      65.363636
                     Kosovo       63.000000
                     Lithuania    64.000000
...                                     ...
Śląsk Wrocław        Latvia       66.000000
                     Poland       61.050000
                     Portugal     67.000000
                     Serbia       66.000000
                     Slovakia     69.000000

[5127 rows x 1 columns]




<a id="section52"></a> 
### <font color="#5E5E5E">15.2 Reestructuración de tablas</font>

### <font size=3 color="#5E5E5E"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#5E5E5E"></i> Ejercicio 29</font> 

Crear un *DataFrame* en el que la primera columna sea el club del jugador, la segunda el nombre, la tercera se denomine `Concept` y pueda tomar los valores `Overall` o `Potential`, y la cuarta se denomine `Score` y contenga la cantidad correspondiente al concepto reflejado en la columna anterior. 

In [166]:
df_fifa_29 = df_fifa.melt(
    id_vars=["Club", "Name"],
    value_vars=["Overall", "Potential"],
    var_name="Concept",
    value_name="Score"
)

df_fifa_29

,Club,Name,Concept,Score
0,FC Barcelona,L. Messi,Overall,94
1,Juventus,Cristiano Ronaldo,Overall,94
2,Paris Saint-Germain,Neymar Jr,Overall,92
3,Manchester United,De Gea,Overall,91
4,Manchester City,K. De Bruyne,Overall,91
...,...,...,...,...
36409,Crewe Alexandra,J. Lundstram,Potential,65
36410,Trelleborgs FF,N. Christoffersson,Potential,63
36411,Cambridge United,B. Worman,Potential,67
36412,Tranmere Rovers,D. Walker-Rice,Potential,66


<br>
<div align="right">
<a href="#inicio"><font size=5 color="#B51700"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

---

<div align="right">
<font size=5 color="#B51700"><i class="fa fa-coffee" aria-hidden="true"></i></font>
</div>